# Detecting Negative Company News

**Author:** Mauro Reverberi

**Program:** MSc AI, Udacity Institute of AI & Technology / Woolf

**Project:** Capstone Project 4, Deep Learning Systems

**Dataset:** Financial PhraseBank v1.0 (Malo et al., 2014), sentences from English financial news about companies, labeled by five to eight of 16 annotators with a background in finance, licensed CC BY-NC-SA 3.0:
https://huggingface.co/datasets/takala/financial_phrasebank

In this notebook I fine-tune a pretrained Transformer (DistilBERT) in PyTorch to read one sentence of company news and decide whether it is negative, neutral or positive for an investor. The output is one probability per class.

**Research question:** How well can a fine-tuned Transformer recognize negative news about a company, and does a loss that gives the rare negative class more weight help to find more of it?

I picked this problem because it connects to my earlier capstone projects. Project 1 built a cleaned dataset of Swiss legal entities from the GLEIF register, Project 2 analyzed new company registrations and Project 3 turned a bankruptcy score into a screening decision. A later capstone project will build a due diligence agent that checks a company, and one standard step of such a check is adverse media screening, reading the news about a company and flagging the negative items. In the workflow I have in mind an analyst reads the flagged items, and I assume that missing a relevant negative item costs more than reviewing an extra one. The labels measure the view of an investor, so a negative sentence here is not yet adverse media in the compliance sense.

## Problem definition and dataset

**Task type:** text classification with a Transformer, one label out of three per sentence. I fine-tune the pretrained encoder `distilbert-base-uncased` with my own classification head in PyTorch.

**Dataset:** Malo et al. (2014) collected sentences from English financial news about companies, many of them Finnish. Between five and eight annotators with a background in finance labeled every sentence as negative, neutral or positive from an investor's perspective. The release has four files with more than 50%, 66%, 75% and full agreement. I use all sentences with more than 50% agreement and keep the agreement level as an extra column (50-65%, 66-74%, 75-99%, 100%), so I can check later whether the model has more trouble where the annotators did not agree. The sentences are real news text, nothing is synthetic or AI-generated, and the license allows academic, non-commercial use. The source was not used in my earlier capstone projects.

4,836 sentences are few for training a Transformer from scratch, but enough to fine-tune a pretrained one. Encoders like BERT are often fine-tuned on tasks with only a few thousand examples, and my training split has 3,393 sentences.

`prepare_dataset.py` downloads the archive from a fixed revision of the Hugging Face repository, removes repeated and contradictory sentences and adds the agreement level. Sentences of the same template with other amounts or years go into one group (TF-IDF similarity of at least 0.65, lower than the 0.8 of the check below, because weights fitted on all sentences give lower values), and the script splits the groups about 70/15/15 into train, validation and test, stratified by label and agreement level. The result `phrasebank.csv` is included, I downloaded the archive on 2026-10-01.

## Setup

I use CUDA if there is a GPU, otherwise the CPU. Seed 42 is the main seed of the notebook.

In [6]:
import copy
import os
import random
import re
import time

# only needed on an NVIDIA GPU, there the deterministic algorithms need a fixed cuBLAS workspace
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import huggingface_hub
import transformers

In [7]:
sns.set_theme(style="whitegrid")
transformers.logging.set_verbosity_error()
transformers.logging.disable_progress_bar()
huggingface_hub.logging.set_verbosity_error()
os.environ["TOKENIZERS_PARALLELISM"] = "false"
torch.use_deterministic_algorithms(True)

In [8]:
MAIN_SEED = 42

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

device: cpu


In [10]:
def set_seed(seed):
    """Set the seed of random, NumPy and PyTorch, so a run can be repeated."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(MAIN_SEED)